In [116]:
using Pkg
Pkg.activate(".")
using Distributed

  Activating project at `c:\Users\mikul\Desktop\Persistence of Shocks\tvPersistence.jl`


In [124]:
num_workers = 4            # ← set to number of CPU cores you want to use
num_replicates = 20        # ← set as desired
addprocs(num_workers)

4-element Vector{Int64}:
 15
 16
 17
 18

In [125]:
# (these were read from example_config.txt in the original script)
@everywhere begin
    data_file                    = "median_RV.csv"
    data_column                  = "x1"
    scale_multiplier             = 1.0
    missingstring                = "NA"

    ar_order                     = 1
    in_sample_window_size        = 1000
    forecast_horizon             = 1
    forecast_length              = 2130
    random_seed                  = 1234

    smoothing_bandwidth          = 0.05
    cutoff_start_index           = 100

    benchmark_method             = "HAR"
    comparison_method            = "tvEWD"

    tvp_kernel_width             = 0.4
    kernel_type                  = "triweight"
    max_ar_order                 = 1
    jmax_scale                   = 5
    ar_lag_for_trend             = 1
    tvp_constant_kernel_width    = 0.1
    irf_kernel_width             = 0.2
    forecast_kernel_width        = 0.5
    smoothing_kernel             = "triweight"
    kernel_type_tvEWD            = "triweight"
    kernel_type_tvHAR            = "triweight"
    kernel_type_tvAR             = "triweight"

    alpha_level                  = 0.05
    verbose_output               = true
end

In [126]:
using CSV, DataFrames, BSON
# launch worker processes

# load essentials on each worker
@everywhere using Random, Statistics
@everywhere include("bootstrap_thresholds.jl")

      From worker 17:	WARNING: replacing module tvOLS_estimator.
      From worker 18:	WARNING: replacing module tvOLS_estimator.
      From worker 16:	WARNING: replacing module tvOLS_estimator.
      From worker 15:	WARNING: replacing module tvOLS_estimator.
      From worker 15:	WARNING: replacing module tvOLS_estimator.
      From worker 16:	WARNING: replacing module tvOLS_estimator.
      From worker 18:	WARNING: replacing module tvOLS_estimator.
      From worker 15:	WARNING: replacing module tvOLS_estimator.
      From worker 15:	WARNING: replacing module tvOLS_estimator.
      From worker 15:	WARNING: replacing module tvOLS_estimator.
      From worker 15:	WARNING: replacing module tvOLS_estimator.
      From worker 15:	WARNING: replacing module tvOLS_estimator.
      From worker 16:	WARNING: replacing module tvOLS_estimator.
      From worker 16:	WARNING: replacing module tvOLS_estimator.
      From worker 16:	WARNING: replacing module tvOLS_estimator.
      From worker 16:	WAR

In [127]:
# load
df = CSV.File(data_file, missingstring=[missingstring], header=true) |> DataFrame;

# turn column name into a Symbol, drop missings & scale
col_sym = Symbol(data_column);
series  = scale_multiplier .* Float64.(df[.!ismissing.(df[!, col_sym]), col_sym]);

      From worker 17:	WARNING: replacing module tvOLS_estimator.
      From worker 17:	WARNING: replacing module tvOLS_estimator.


In [128]:
sed_vals = pmap(1:num_replicates) do i
    # re-seed for reproducibility
    Random.seed!(random_seed + i)

    calculate_bootstrap_threshold_parallel_V2(
        i, series,
        ar_order, in_sample_window_size, forecast_horizon,
        smoothing_bandwidth,
        Symbol(benchmark_method), Symbol(comparison_method);
        fcast_len                  = forecast_length,
        tvp_kernel_width           = tvp_kernel_width,
        kernel_type_tvEWD          = kernel_type_tvEWD,
        kernel_type_tvHAR          = kernel_type_tvHAR,
        kernel_type_tvAR           = kernel_type_tvAR,
        smoothing_kernel           = smoothing_kernel,
        max_ar_order               = max_ar_order,
        jmax_scale                 = jmax_scale,
        ar_lag_for_trend           = ar_lag_for_trend,
        tvp_constant_kernel_width  = tvp_constant_kernel_width,
        irf_kernel_width           = irf_kernel_width,
        forecast_kernel_width      = forecast_kernel_width
    )
end;

# remove working processes
rmprocs(workers())

      From worker 17:	[ Info: Performing boostrap simulation number 3
      From worker 16:	[ Info: Performing boostrap simulation number 2
      From worker 15:	[ Info: Performing boostrap simulation number 1
      From worker 18:	[ Info: Performing boostrap simulation number 4
      From worker 16:	[ Info: Bootstrap 2 generated.
      From worker 16:	[ Info: Performing boostrap simulation number 5
      From worker 17:	[ Info: Bootstrap 3 generated.
      From worker 17:	[ Info: Performing boostrap simulation number 6
      From worker 15:	[ Info: Bootstrap 1 generated.
      From worker 18:	[ Info: Bootstrap 4 generated.
      From worker 15:	[ Info: Performing boostrap simulation number 7
      From worker 18:	[ Info: Performing boostrap simulation number 8
      From worker 17:	[ Info: Bootstrap 6 generated.
      From worker 17:	[ Info: Performing boostrap simulation number 9
      From worker 16:	[ Info: Bootstrap 5 generated.
      From worker 16:	[ Info: Performing boostrap si

Task (done) @0x0000011f2d9eeca0

In [129]:
sed_vals

20-element Vector{Vector{Float64}}:
 [-5.1791839111421907e-5, -4.878865924597512e-5, -4.59694937669722e-5, -4.3335198062910776e-5, -4.0886051689154714e-5, -3.862158525090186e-5, -3.654052545662434e-5, -3.4640734122292775e-5, -3.29193374525788e-5, -3.1372055845071674e-5  …  -6.751690044498783e-5, -6.639757617340987e-5, -6.51670939037476e-5, -6.38216701037e-5, -6.235740275199579e-5, -6.07701944315359e-5, -5.905579325233103e-5, -5.7209773487494446e-5, -5.5227548411706595e-5, -5.31044345101104e-5]
 [8.02597683854974e-5, 7.621934913838981e-5, 7.19777982623479e-5, 6.754545449450433e-5, 6.293325429405504e-5, 5.815229578684e-5, 5.321356509121161e-5, 4.8128207733317416e-5, 4.290728914759074e-5, 3.756157169196195e-5  …  -1.1510728314703417e-5, -9.70387589625853e-6, -7.85622413654579e-6, -5.968581577661756e-6, -4.041774594194542e-6, -2.076652856345968e-6, -7.407927490096083e-8, 1.9650580783403537e-6, 4.039854550030517e-6, 6.149404583845591e-6]
 [-8.810941496273556e-6, -8.157882727165392e-6, -7.51

In [132]:
# 1) Count NaNs in each inner vector
nan_counts_per_series = map(v -> count(isnan, v), sed_vals)

# 2) Total number of NaNs across all series
total_nans = sum(nan_counts_per_series)

426

In [134]:
using StatsBase  # for quantile
function compute_global_threshold_V2(
        list_of_sed_vectors::Vector{Vector{Float64}},
        cutoff_start_index::Int,
        alpha_level::Float64 = 0.05
    )::Float64

    B = length(list_of_sed_vectors)
    out_of_sample_length = length(list_of_sed_vectors[1])
    time_cutoffs = Float64[]

    for t in cutoff_start_index:out_of_sample_length
        # collect the B values at time t
        raw_vals = [ list_of_sed_vectors[b][t] for b in 1:B ]
        # drop any NaNs
        clean_vals = filter(!isnan, raw_vals)

        # if everything was NaN you might skip or push a NaN
        if isempty(clean_vals)
            continue
        end

        push!(time_cutoffs, quantile(clean_vals, 1 - alpha_level))
    end

    return median(time_cutoffs)
end


compute_global_threshold_V2 (generic function with 2 methods)

In [136]:
sed_vals

20-element Vector{Vector{Float64}}:
 [-5.1791839111421907e-5, -4.878865924597512e-5, -4.59694937669722e-5, -4.3335198062910776e-5, -4.0886051689154714e-5, -3.862158525090186e-5, -3.654052545662434e-5, -3.4640734122292775e-5, -3.29193374525788e-5, -3.1372055845071674e-5  …  -6.751690044498783e-5, -6.639757617340987e-5, -6.51670939037476e-5, -6.38216701037e-5, -6.235740275199579e-5, -6.07701944315359e-5, -5.905579325233103e-5, -5.7209773487494446e-5, -5.5227548411706595e-5, -5.31044345101104e-5]
 [8.02597683854974e-5, 7.621934913838981e-5, 7.19777982623479e-5, 6.754545449450433e-5, 6.293325429405504e-5, 5.815229578684e-5, 5.321356509121161e-5, 4.8128207733317416e-5, 4.290728914759074e-5, 3.756157169196195e-5  …  -1.1510728314703417e-5, -9.70387589625853e-6, -7.85622413654579e-6, -5.968581577661756e-6, -4.041774594194542e-6, -2.076652856345968e-6, -7.407927490096083e-8, 1.9650580783403537e-6, 4.039854550030517e-6, 6.149404583845591e-6]
 [-8.810941496273556e-6, -8.157882727165392e-6, -7.51

In [137]:
cutoff_start_index

100

In [142]:
thr = compute_global_threshold_V2(sed_vals, cutoff_start_index, alpha_level)
println("SED threshold: ", thr)

SED threshold: -8.473937894274338e-6


In [133]:
# will create sed_thresholds.bson in your working directory
BSON.@save "sed_thresholds.bson" sed_vals thr